[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wahyudi-Satriawan/244107020137_TI3H_PembelajaranMesin/blob/main/Jobsheet%20Dataset%20PBL/JS04.ipynb)

# JOBSHEET 04: KLASTERISASI (K-MEANS & DBSCAN)
### Segmentasi Pasar Properti Multi-Kota Indonesia, Penentuan K Optimal Berbasis SSE & Silhouette Sampling, serta Klasterisasi Spasial Berbasis Kepadatan pada Dataset PBL RumahPoint
**Mata Kuliah:** Pembelajaran Mesin — Semester 5 (Tahun Akademik 2025/2026)  
**Mahasiswa:** Wahyudi Satriawan Hamid  
**NIM / Kelas:** 244107020137 / TI-3H  
**Program Studi:** D-IV Teknik Informatika — Jurusan Teknologi Informasi — Politeknik Negeri Malang  

---

## Ringkasan Eksekutif & Peta Adaptasi Modul ke Dataset PBL
Jobsheet 04 berfokus pada penguasaan dua paradigma utama pembelajaran mesin tanpa pengawasan (*unsupervised learning*):
1. **Klasterisasi Berbasis Partisi Pusat (*Centroid-Based Partitioning*):** Algoritma **K-Means**, optimasi $K$ melalui metode *Elbow* (Sum of Squared Errors / SSE), dan evaluasi kualitas pemisahan klaster (*Silhouette Score* & *Davies-Bouldin Index*).
2. **Klasterisasi Berbasis Kepadatan Spasial (*Density-Based Spatial Clustering*):** Algoritma **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)**, penentuan parameter $\varepsilon$ (*eps*) berbasis kurva *k-Distance Graph*, dan identifikasi data anomali (*noise points*).

Sesuai dengan kontrak teknis pada `DECISION_LOG.md`, seluruh konsep inti modul Polinema ini diimplementasikan secara langsung pada dataset riil berskala industri (**`Dataset/properti_multi_kota_clean.csv`**, 56.170 baris data terverifikasi):

| Bagian Modul | Topik & Metode Asli Modul | Adaptasi & Implementasi Dataset PBL | Justifikasi Metodologis & Rekayasa Komputasi |
|:---|:---|:---|:---|
| **Bagian 1** | **Persiapan & Normalisasi Fitur** (Iris, Blobs, Digits) | Pemuatan data bersih `properti_multi_kota_clean.csv` (56.170 baris), rekayasa fitur `log1p_harga` & `is_komersil`, standardisasi fitur via `Pipeline` scikit-learn, serta reduksi dimensi PCA 2D untuk ruang inspeksi visual. | Mencegah sebaran harga miring mendominasi jarak Euclidean; menghindari *curse of dimensionality* dengan tidak meng-OHE kolom lokasi (disimpan untuk profiling). |
| **Bagian 2** | **Praktikum 1 & 2:** K-Means Clustering, Elbow SSE, & Evaluasi (Iris, Blobs, Digits) | K-Means clustering pada seluruh populasi (56.170 baris) untuk rentang $K = 2..10$, perhitungan SSE (Inertia), Davies-Bouldin Index (DBI), dan Silhouette Score dengan **sampling terkontrol ($N=5.000$)**. Profiling domain lengkap & validasi eksternal ARI terhadap `kelas_harga`. | Mitigasi crash *Out-of-Memory* (OOM) alokasi matriks jarak ~25 GB di Google Colab. Membuktikan struktur segmentasi pasar properti komersial vs residensial. |
| **Bagian 3** | **Praktikum 3 & Tugas 2:** DBSCAN Clustering & Eksperimen Parameter (Blobs, Moons) | Subsampling terstratifikasi 10.000 baris (`kelas_harga`), penentuan parameter `eps` ilmiah melalui **k-Distance Graph** ($k=10$), uji grid kombinasi `eps` persentil vs `min_samples` [10, 20, 30] dengan guard evaluasi non-noise, dan visualisasi klaster + noise pada PCA 2D. | Mengatasi kompleksitas waktu/memori $\mathcal{O}(N^2)$ DBSCAN. Menghilangkan asumsi tebakan subjektif parameter `eps` berbasis kerapatan spasial riil. |
| **Bagian 4** | **Komparasi K-Means vs DBSCAN & Profiling Noise** | Perbandingan langsung K-Means vs DBSCAN pada 10.000 baris yang identik: tabel metrik (Silhouette, DBI, ARI, Noise %), scatter plot 2D berdampingan, dan inspeksi karakteristik titik Noise `-1`. | Memberikan pemahaman komparatif mendalam: K-Means memaksa partisi bola (*spherical*), sedangkan DBSCAN menangkap klaster berdensitas tinggi dan memisahkan properti mewah ekstrem/komersil sebagai noise. |

## Justifikasi Ilmiah & Metodologis Adaptasi Modul

### 1. Fokus Langsung pada Dataset PBL RumahPoint (`properti_multi_kota_clean.csv`)
Praktikum bawaan modul Polinema menggunakan dataset sintetis atau mainan seperti Iris (150 sampel), Blobs (300 sampel), Digits (1.797 sampel), dan make_moons (1.000 sampel). Seluruh praktikum dasar tersebut telah dikerjakan dan didokumentasikan secara lengkap pada arsip modul standar (`Jobsheet Sesuai Modul/JS04.ipynb`).

Pada notebook ini, fokus 100% dialihkan ke **`Dataset/properti_multi_kota_clean.csv` (56.170 baris)** untuk menguji skalabilitas, ketahanan numerik, dan interpretasi domain bisnis properti riil.

### 2. Pemilihan Fitur Jarak & Retensi Non-OHE Lokasi (Anti-Curse of Dimensionality)
Klasterisasi bergantung penuh pada perhitungan metrik jarak (Euclidean). Fitur yang dipilih sebagai matriks jarak berdimensi 5:
$$\mathbf{X} = [\text{luas\_m2}, \text{luas\_bangunan}, \text{jumlah\_kamar}, \text{jumlah\_kamar\_mandi}, \text{log1p\_harga}]$$
- **Transformasi $\log(1+\text{harga})$ (`np.log1p`):** Distribusi harga properti memiliki kemiringan positif ekstrem (*heavy right-skewed*), dengan rentang puluhan juta hingga ratusan miliar Rupiah. Tanpa transformasi logaritmik, variansi harga akan mendominasi kuadrat selisih jarak Euclidean, sehingga klasterisasi hanya akan mengelompokkan harga dan mengabaikan karakteristik fisik (luas dan kamar).
- **Retensi Non-OHE Lokasi (588 Kategori Unik):** Kolom `lokasi_bersih` dan `sumber` **TIDAK** di-One-Hot Encoding ke dalam ruang jarak. Menambahkan 588 kolom sparse biner akan memicu *curse of dimensionality* (fenomena konsentrasi jarak di mana rasio jarak terjauh dan terdekat mendekati 1, membuat jarak Euclidean kehilangan makna diferensiasi). Kolom lokasi dan jenis properti disimpan secara khusus sebagai metadata untuk **profiling domain isi klaster** pasca-klasterisasi.

### 3. Keunggulan Data Bersih Terdeduplikasi untuk Validitas Kepadatan Spasial DBSCAN
Algoritma DBSCAN dan metode berbasis densitas lainnya mendefinisikan klaster berdasarkan kerapatan jumlah titik dalam radius $\varepsilon$. Pada dataset mentah, terdapat 7.156 baris duplikasi persis akibat scraping ganda. Keberadaan duplikat identik ini akan menghasilkan jarak $0.0$ antar baris, yang secara artifisial menciptakan kantong kepadatan palsu (*false core points / micro-clusters*). Menggunakan `properti_multi_kota_clean.csv` yang telah dideduplikasi menjamin estimasi kepadatan spasial mencerminkan realitas pasar properti yang sesungguhnya.

---
## Bagian 1 — Pemuatan Data & Pipeline Preprocessing Unsupervised

Pada tahap ini, kita memuat dataset bersih `properti_multi_kota_clean.csv` menggunakan skema pemuat cerdas dual-mode, menyiapkan fitur numerik input dan variabel indikator bisnis untuk profiling, mengonstruksi `Pipeline` unsupervised (imputasi median + standardisasi z-score), serta memproyeksikan fitur ke ruang 2 dimensi menggunakan **Principal Component Analysis (PCA)** untuk keperluan visualisasi.

In [ ]:
# ==============================================================================
# BAGIAN 1.1: PEMUATAN DATASET DUAL-MODE (LOCAL PATH & FALLBACK GITHUB RAW)
# ==============================================================================
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi gaya visualisasi standar
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 100

DATASET_FILENAME = "properti_multi_kota_clean.csv"
LOCAL_PATH = os.path.join("..", "Dataset", DATASET_FILENAME)
GITHUB_RAW_URL = f"https://raw.githubusercontent.com/Wahyudi-Satriawan/244107020137_TI3H_PembelajaranMesin/main/Dataset/{DATASET_FILENAME}"

if os.path.exists(LOCAL_PATH):
    print(f"[LOADER] Memuat dataset dari penyimpanan lokal: {LOCAL_PATH}")
    df = pd.read_csv(LOCAL_PATH)
else:
    print(f"[LOADER] Path lokal tidak ditemukan. Mengunduh langsung dari GitHub: {GITHUB_RAW_URL}")
    df = pd.read_csv(GITHUB_RAW_URL)

print(f"[LOADER] Berhasil memuat data: {df.shape[0]:,} baris, {df.shape[1]} kolom.")
df.head()

In [ ]:
# ==============================================================================
# BAGIAN 1.2: PREPARASI FITUR JARAK DAN FITUR PROFILING BISNIS
# ==============================================================================
# 1. Transformasi logaritmik harga untuk menstabilkan variansi skala
df['log1p_harga'] = np.log1p(df['harga'])

# 2. Fitur indikator bisnis komersial (kamar == 0) khusus untuk profiling domain
df['is_komersil'] = (df['jumlah_kamar'] == 0).astype(int)

# 3. Fitur jarak kuantitatif (5 dimensi)
FITUR_JARAK = ['luas_m2', 'luas_bangunan', 'jumlah_kamar', 'jumlah_kamar_mandi', 'log1p_harga']

print("=" * 70)
print("AUDIT FITUR INPUT KLASTERISASI")
print("=" * 70)
print(f"Fitur Jarak ({len(FITUR_JARAK)} fitur): {FITUR_JARAK}")
print(f"Total Sampel Properti Bersih: {len(df):,} unit")
print(f"Proporsi Properti Komersial (kamar=0): {df['is_komersil'].mean() * 100:.2f}% ({df['is_komersil'].sum():,} unit)")
print("-" * 70)
print(df[FITUR_JARAK].describe().T[['count', 'mean', 'std', 'min', '50%', 'max']])

In [ ]:
# ==============================================================================
# BAGIAN 1.3: PIPELINE PREPROCESSING UNSUPERVISED & PROYEKSI PCA 2D
# ==============================================================================
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Konstruksi pipeline unsupervised
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Fit dan transform seluruh populasi data (56.170 baris)
X_scaled = preprocessor.fit_transform(df[FITUR_JARAK])

# Reduksi dimensi PCA 2 komponen untuk visualisasi ruang fitur
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

var_ratio = pca.explained_variance_ratio_
print("=" * 70)
print("ANALISIS REDUKSI DIMENSI PRINCIPAL COMPONENT ANALYSIS (PCA)")
print("=" * 70)
print(f"Variansi PC1: {var_ratio[0] * 100:.2f}%")
print(f"Variansi PC2: {var_ratio[1] * 100:.2f}%")
print(f"Total Variansi Kumulatif PC1 & PC2: {np.sum(var_ratio) * 100:.2f}%")
print("-" * 70)

# Bobot Loading Fitur pada PC1 dan PC2
df_loadings = pd.DataFrame(
    pca.components_.T,
    index=FITUR_JARAK,
    columns=['PC1_Loading', 'PC2_Loading']
)
print("Matriks Loading Fitur pada Komponen Utama:")
print(df_loadings.round(4))
print("=" * 70)

---
## Bagian 2 — K-Means Clustering & Penentuan K Optimal (Adaptasi Praktikum 1 & 2)

Pada bagian ini, kita menerapkan algoritma **K-Means** (`sklearn.cluster.KMeans`) pada seluruh 56.170 baris data:
1. **Pencarian Jumlah Klaster Optimal ($K = 2..10$):** Menghitung nilai **Inertia (SSE)** untuk metode *Elbow*.
2. **Evaluasi Koefisien Pemisahan (*Silhouette Score* & *Davies-Bouldin Index*):**
   > [!IMPORTANT]
   > Menghitung matriks jarak Silhouette berpasangan untuk 56.170 baris secara penuh membutuhkan alokasi memori sebesar $56.170^2 \times 8 \text{ byte} \approx 25,2 \text{ GB RAM}$, yang pasti memicu crash *Out-of-Memory* (OOM) pada RAM Google Colab (~12 GB). Oleh karena itu, kita **WAJIB** memanfaatkan parameter `sample_size=5000` dengan `random_state=42` pada `silhouette_score` guna menghasilkan estimasi koefisien yang stabil, akurat, dan ramah komputasi.
3. **Pemilihan $K$ Optimal & Pembanding Struktural ($K=3$):** Memilih $K$ terbaik serta membandingkannya secara struktural dengan 3 jenjang `kelas_harga`.
4. **Visualisasi Ruang 2D & Profiling Domain:** Memproyeksikan klaster dan centroid ke bidang PCA serta membedah profil bisnis tiap klaster.

In [ ]:
# ==============================================================================
# BAGIAN 2.1: EKSPERIMEN K-MEANS K=2..10 & METRIK EVALUASI UNSUPERVISED
# ==============================================================================
import time
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

k_range = range(2, 11)
sse_list = []
silhouette_list = []
dbi_list = []
durasi_list = []

print("=" * 75)
print("MEMULAI PENCARIAN K OPTIMAL K-MEANS (DATASET PENUH: 56.170 BARIS)")
print("=" * 75)
print(f"{'K':<4} | {'SSE (Inertia)':<15} | {'Silhouette (N=5k)':<18} | {'DBI Score':<12} | {'Durasi (s)':<10}")
print("-" * 75)

for k in k_range:
    t_start = time.time()
    kmeans_model = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = kmeans_model.fit_predict(X_scaled)
    t_elapsed = time.time() - t_start
    
    # 1. Sum of Squared Errors (SSE / Inertia)
    sse = kmeans_model.inertia_
    sse_list.append(sse)
    
    # 2. Silhouette Score dengan subsample 5.000 titik (Anti-OOM)
    sil = silhouette_score(X_scaled, labels, sample_size=5000, random_state=42)
    silhouette_list.append(sil)
    
    # 3. Davies-Bouldin Index (Semakin rendah semakin terpisah dengan baik)
    dbi = davies_bouldin_score(X_scaled, labels)
    dbi_list.append(dbi)
    
    durasi_list.append(t_elapsed)
    print(f"{k:<4} | {sse:<15.2f} | {sil:<18.4f} | {dbi:<12.4f} | {t_elapsed:<10.2f}")

print("=" * 75)
df_eval_kmeans = pd.DataFrame({
    'K': list(k_range),
    'SSE': sse_list,
    'Silhouette_Score': silhouette_list,
    'Davies_Bouldin_Index': dbi_list,
    'Durasi_Detik': durasi_list
})

In [ ]:
# ==============================================================================
# BAGIAN 2.2: VISUALISASI METODE ELBOW DAN KURVA SILHOUETTE SCORE
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Kurva Elbow (SSE vs K)
axes[0].plot(k_range, sse_list, marker='o', color='#1f77b4', linewidth=2, markersize=7)
axes[0].set_title("Kurva Elbow K-Means (SSE vs K)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Jumlah Klaster (K)", fontsize=11)
axes[0].set_ylabel("Sum of Squared Errors (SSE / Inertia)", fontsize=11)
axes[0].set_xticks(k_range)
axes[0].grid(True, linestyle='--', alpha=0.7)

# Panel 2: Kurva Silhouette Score vs K
axes[1].plot(k_range, silhouette_list, marker='s', color='#2ca02c', linewidth=2, markersize=7)
axes[1].set_title("Kurva Silhouette Score K-Means (Subsample N=5.000)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Jumlah Klaster (K)", fontsize=11)
axes[1].set_ylabel("Silhouette Score (Makin Tinggi Makin Baik)", fontsize=11)
axes[1].set_xticks(k_range)
axes[1].grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# BAGIAN 2.3: PEMILIHAN K OPTIMAL TERPROGRAM & PEMODELAN K-MEANS
# ==============================================================================
# Seleksi K optimal berdasarkan Silhouette Score tertinggi
idx_best = int(np.argmax(silhouette_list))
k_optimal = list(k_range)[idx_best]

print(f"[REKOMENDASI] K optimal terprogram berdasarkan Silhouette Score tertinggi: K = {k_optimal}")
print(f"              Silhouette Score = {silhouette_list[idx_best]:.4f}, DBI = {dbi_list[idx_best]:.4f}")

# Kita latih model K-Means final menggunakan K optimal
kmeans_final = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
df['cluster_kmeans'] = kmeans_final.fit_predict(X_scaled)

# Latih pula K=3 sebagai pembanding struktural terhadap 3 kelas harga (Murah, Sedang, Mahal)
kmeans_k3 = KMeans(n_clusters=3, n_init=10, random_state=42)
df['cluster_k3'] = kmeans_k3.fit_predict(X_scaled)

print(f"[INFO] Pelatihan K-Means selesai: Klaster tersimpan di kolom 'cluster_kmeans' (K={k_optimal}) dan 'cluster_k3' (K=3).")

In [ ]:
# ==============================================================================
# BAGIAN 2.4: VISUALISASI SEBARAN KLASTER K-MEANS PADA RUANG PCA 2D
# ==============================================================================
# Transformasikan pusat klaster (centroid) ke ruang koordinat PCA 2D
centroids_pca = pca.transform(kmeans_final.cluster_centers_)

plt.figure(figsize=(10, 6))
scatter = plt.scatter(
    X_pca[:, 0], X_pca[:, 1],
    c=df['cluster_kmeans'],
    cmap='tab10',
    alpha=0.35,
    s=12,
    edgecolors='none'
)
plt.scatter(
    centroids_pca[:, 0], centroids_pca[:, 1],
    c='red',
    marker='X',
    s=250,
    edgecolors='black',
    linewidths=2,
    label='Centroid Klaster'
)
plt.title(f"Visualisasi Klaster K-Means (K={k_optimal}) pada Ruang PCA 2D (56.170 Properti)", fontsize=13, fontweight='bold')
plt.xlabel(f"Principal Component 1 ({var_ratio[0]*100:.1f}% Variansi)", fontsize=11)
plt.ylabel(f"Principal Component 2 ({var_ratio[1]*100:.1f}% Variansi)", fontsize=11)
plt.colorbar(scatter, label='ID Klaster')
plt.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.show()

### Profiling Domain Klaster K-Means & Validasi Eksternal Adjusted Rand Index (ARI)

> [!NOTE]
> **Aturan Evaluasi Tanpa Pengawasan (*Unsupervised Ground Rule*):**
> Kolom target `kelas_harga` (`Murah`, `Sedang`, `Mahal`) **SAMA SEKALI TIDAK DIGUNAKAN** selama proses pelatihan klasterisasi K-Means. `kelas_harga` diuji secara eksklusif menggunakan **Adjusted Rand Index (ARI)** pasca-pelatihan sebagai metrik validasi eksternal guna mengevaluasi sejauh mana segmentasi morfologi fisik (luas & kamar) dan harga logaritmik secara alami berkorespondensi dengan jenjang harga pasar.

In [ ]:
# ==============================================================================
# BAGIAN 2.5: PROFILING DOMAIN KLASTER K-MEANS & VALIDASI EKSTERNAL ARI
# ==============================================================================
from sklearn.metrics import adjusted_rand_score

# Validasi eksternal ARI terhadap target kelas_harga
ari_kopt = adjusted_rand_score(df['kelas_harga'], df['cluster_kmeans'])
ari_k3 = adjusted_rand_score(df['kelas_harga'], df['cluster_k3'])

print("=" * 75)
print("VALIDASI EKSTERNAL: ADJUSTED RAND INDEX (ARI) TERHADAP 'kelas_harga'")
print("=" * 75)
print(f"ARI Klaster K-Means Optimal (K={k_optimal}) vs kelas_harga: {ari_kopt:.4f}")
print(f"ARI Klaster K-Means Struktur (K=3)       vs kelas_harga: {ari_k3:.4f}")
print("=" * 75)

# Tabel Profiling Domain per Klaster
profil_list = []
for c_id in sorted(df['cluster_kmeans'].unique()):
    sub = df[df['cluster_kmeans'] == c_id]
    top_lokasi = ", ".join(sub['lokasi_bersih'].value_counts().head(3).index.tolist())
    dist_kelas = sub['kelas_harga'].value_counts(normalize=True).to_dict()
    kelas_str = f"M:{dist_kelas.get('Murah', 0)*100:.0f}% S:{dist_kelas.get('Sedang', 0)*100:.0f}% H:{dist_kelas.get('Mahal', 0)*100:.0f}%"
    
    profil_list.append({
        'Cluster': f"Klaster {c_id}",
        'Jumlah_Unit': len(sub),
        'Persen_Total': f"{len(sub)/len(df)*100:.1f}%",
        'Median_Harga_Rp': f"Rp {sub['harga'].median():,.0f}",
        'Median_Luas_Tanah_m2': f"{sub['luas_m2'].median():.1f}",
        'Median_Luas_Bgn_m2': f"{sub['luas_bangunan'].median():.1f}",
        'Median_Kamar': f"{sub['jumlah_kamar'].median():.0f}",
        'Persen_Komersil': f"{sub['is_komersil'].mean()*100:.1f}%",
        'Dominasi_Kelas_Harga': kelas_str,
        'Top_3_Lokasi': top_lokasi
    })

df_profil_kmeans = pd.DataFrame(profil_list)
print("\nTABEL PROFILING DOMAIN KLASTER K-MEANS:")
print(df_profil_kmeans.to_string(index=False))

---
## Bagian 3 — DBSCAN Clustering & Penentuan `eps` Berbasis k-Distance Graph (Adaptasi Praktikum 3 & Tugas)

Pada bagian ini, kita menerapkan **DBSCAN (Density-Based Spatial Clustering of Applications with Noise)**:
1. **Stratified Subsampling (10.000 Baris):**
   > [!NOTE]
   > Algoritma DBSCAN menghitung jarak ketetanggaan berpasangan dengan kompleksitas waktu dan ruang $\mathcal{O}(N^2)$. Menjalankan DBSCAN pada 56.170 baris secara simultan akan memakan waktu komputasi yang sangat lama serta alokasi memori berlebih. Berdasarkan kontrak teknis pada `DECISION_LOG.md`, kita mengambil subsampel terstratifikasi sebanyak **10.000 baris** berdasarkan `kelas_harga` (`random_state=42`).
2. **Penentuan Parameter `eps` Ilmiah (Anti-Asumsi):**
   Alih-alih menebak nilai `eps` secara subjektif, kita menggunakan metode kurva **k-Distance Graph** (`NearestNeighbors(n_neighbors=10)`, aturan praktis $min\_samples = 2 \times D = 10$ untuk $D=5$ fitur). Jarak tetangga ke-$k$ diurutkan secara menaik untuk menemukan titik siku (*knee*) dan kandidat persentil (P85, P90, P95, P98).
3. **Eksperimen Grid Parameter DBSCAN:**
   Menguji variasi kandidat `eps` dan `min_samples` (`[10, 20, 30]`). Metrik Silhouette dan DBI dihitung khusus pada sampel non-noise dengan **guard ketat** (`if n_clusters >= 2 and n_non_noise > n_clusters`).

In [ ]:
# ==============================================================================
# BAGIAN 3.1: STRATIFIED SUBSAMPLING (10.000 BARIS) UNTUK DBSCAN
# ==============================================================================
from sklearn.model_selection import StratifiedShuffleSplit

# Subsampling 10.000 baris terstratifikasi terhadap kelas_harga
sss = StratifiedShuffleSplit(n_splits=1, test_size=10000, random_state=42)
for _, sub_idx in sss.split(df, df['kelas_harga']):
    df_sub = df.iloc[sub_idx].copy().reset_index(drop=True)

# Transformasikan subsample dengan pipeline yang sama
X_scaled_sub = preprocessor.transform(df_sub[FITUR_JARAK])
X_pca_sub = pca.transform(X_scaled_sub)

print("=" * 70)
print(f"SUBSAMPLE TERSTRATIFIKASI DBSCAN: {len(df_sub):,} baris")
print("Distribusi Target kelas_harga pada Subsample:")
print(df_sub['kelas_harga'].value_counts(normalize=True).round(4) * 100)
print("=" * 70)

In [ ]:
# ==============================================================================
# BAGIAN 3.2: PENENTUAN EPS BERBASIS K-DISTANCE GRAPH (NEAREST NEIGHBORS)
# ==============================================================================
from sklearn.neighbors import NearestNeighbors

# Aturan praktis: k = 2 * D = 2 * 5 = 10
k_neighbors = 10
nbrs = NearestNeighbors(n_neighbors=k_neighbors, metric='euclidean')
nbrs.fit(X_scaled_sub)
distances, indices = nbrs.kneighbors(X_scaled_sub)

# Ambil jarak ke tetangga ke-k (kolom terakhir) dan urutkan menaik
k_distances = np.sort(distances[:, -1])

# Hitung kandidat eps pada berbagai persentil distribusi jarak
p_values = [85, 90, 95, 98]
kandidat_eps = [float(np.percentile(k_distances, p)) for p in p_values]

plt.figure(figsize=(10, 5))
plt.plot(k_distances, color='#d62728', linewidth=2, label=f'Jarak Tetangga ke-{k_neighbors}')
for p, eps_val in zip(p_values, kandidat_eps):
    plt.axhline(y=eps_val, linestyle='--', alpha=0.7, label=f'P{p} (eps = {eps_val:.3f})')

plt.title(f"k-Distance Graph (k={k_neighbors}) untuk Penentuan Ilmiah Parameter eps DBSCAN", fontsize=12, fontweight='bold')
plt.xlabel("Indeks Titik (Terurut Menaik)", fontsize=11)
plt.ylabel(f"Jarak Euclidean ke Tetangga ke-{k_neighbors}", fontsize=11)
plt.legend(loc='upper left', frameon=True)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print("Kandidat Nilai eps Berdasarkan Titik Persentil k-Distance:")
for p, val in zip(p_values, kandidat_eps):
    print(f"- Persentil {p}%: eps = {val:.4f}")

In [ ]:
# ==============================================================================
# BAGIAN 3.3: EKSPERIMEN GRID PARAMETER DBSCAN (EPS X MIN_SAMPLES)
# ==============================================================================
from sklearn.cluster import DBSCAN

min_samples_grid = [10, 20, 30]
grid_results = []

print("=" * 85)
print("MEMULAI GRID EXPERIMENT PARAMETER DBSCAN (10.000 SAMPEL PROPERTI)")
print("=" * 85)
print(f"{'eps':<8} | {'min_samp':<8} | {'Klaster':<8} | {'Noise (%)':<12} | {'Silhouette (Non-Noise)':<22} | {'DBI Score':<10}")
print("-" * 85)

for eps_val in kandidat_eps:
    for ms in min_samples_grid:
        db = DBSCAN(eps=eps_val, min_samples=ms)
        labels_db = db.fit_predict(X_scaled_sub)
        
        # Identifikasi klaster dan noise (-1)
        valid_mask = labels_db != -1
        unique_valid = set(labels_db[valid_mask])
        n_clusters = len(unique_valid)
        n_noise = np.sum(~valid_mask)
        noise_ratio = (n_noise / len(labels_db)) * 100
        
        # Guard: Silhouette & DBI hanya valid jika klaster non-noise >= 2 dan n_valid > n_clusters
        n_valid = np.sum(valid_mask)
        if n_clusters >= 2 and n_valid > n_clusters:
            sil_db = silhouette_score(X_scaled_sub[valid_mask], labels_db[valid_mask])
            dbi_db = davies_bouldin_score(X_scaled_sub[valid_mask], labels_db[valid_mask])
        else:
            sil_db = np.nan
            dbi_db = np.nan
            
        grid_results.append({
            'eps': round(eps_val, 4),
            'min_samples': ms,
            'n_clusters': n_clusters,
            'noise_pct': round(noise_ratio, 2),
            'silhouette_non_noise': round(sil_db, 4) if not np.isnan(sil_db) else None,
            'dbi_score': round(dbi_db, 4) if not np.isnan(dbi_db) else None
        })
        
        sil_str = f"{sil_db:.4f}" if not np.isnan(sil_db) else "N/A (<2 cl)"
        dbi_str = f"{dbi_db:.4f}" if not np.isnan(dbi_db) else "N/A"
        print(f"{eps_val:<8.4f} | {ms:<8} | {n_clusters:<8} | {noise_ratio:<12.2f} | {sil_str:<22} | {dbi_str:<10}")

print("=" * 85)
df_dbscan_grid = pd.DataFrame(grid_results)

In [ ]:
# ==============================================================================
# BAGIAN 3.4: PEMILIHAN MODEL DBSCAN TERBAIK & VISUALISASI PCA 2D
# ==============================================================================
# Filter konfigurasi dengan klaster valid >= 2 dan noise wajar (1% - 30%)
df_valid_db = df_dbscan_grid[
    (df_dbscan_grid['n_clusters'] >= 2) & 
    (df_dbscan_grid['noise_pct'] >= 1.0) & 
    (df_dbscan_grid['noise_pct'] <= 30.0) &
    (df_dbscan_grid['silhouette_non_noise'].notnull())
]

if not df_valid_db.empty:
    best_config = df_valid_db.sort_values(by='silhouette_non_noise', ascending=False).iloc[0]
    best_eps = best_config['eps']
    best_ms = int(best_config['min_samples'])
else:
    # Fallback ke konfigurasi P90 & min_samples=10 jika filter ketat tidak terpenuhi
    best_eps = kandidat_eps[1]
    best_ms = 10

print(f"[REKOMENDASI] Konfigurasi DBSCAN Terpilih: eps = {best_eps:.4f}, min_samples = {best_ms}")

# Fit model DBSCAN final pada subsample
dbscan_final = DBSCAN(eps=best_eps, min_samples=best_ms)
df_sub['cluster_dbscan'] = dbscan_final.fit_predict(X_scaled_sub)

n_cl_final = len(set(df_sub['cluster_dbscan'])) - (1 if -1 in df_sub['cluster_dbscan'].values else 0)
n_ns_final = (df_sub['cluster_dbscan'] == -1).sum()
print(f"[HASIL] Jumlah Klaster Terbentuk: {n_cl_final} | Titik Noise: {n_ns_final:,} unit ({n_ns_final/len(df_sub)*100:.2f}%)")

# Visualisasi PCA 2D DBSCAN
plt.figure(figsize=(10, 6))
mask_noise = df_sub['cluster_dbscan'] == -1
mask_core = df_sub['cluster_dbscan'] != -1

# Titik Klaster
plt.scatter(
    X_pca_sub[mask_core, 0], X_pca_sub[mask_core, 1],
    c=df_sub.loc[mask_core, 'cluster_dbscan'],
    cmap='tab10',
    s=15,
    alpha=0.6,
    label='Clustered Points'
)
# Titik Noise (-1) ditampilkan hitam
plt.scatter(
    X_pca_sub[mask_noise, 0], X_pca_sub[mask_noise, 1],
    c='black',
    marker='x',
    s=25,
    alpha=0.5,
    label='Noise / Outlier (-1)'
)
plt.title(f"Visualisasi Klaster DBSCAN (eps={best_eps:.3f}, min_samples={best_ms}) pada PCA 2D", fontsize=12, fontweight='bold')
plt.xlabel(f"PC1 ({var_ratio[0]*100:.1f}%)", fontsize=11)
plt.ylabel(f"PC2 ({var_ratio[1]*100:.1f}%)", fontsize=11)
plt.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.show()

---
## Bagian 4 — Komparasi K-Means vs DBSCAN & Profiling Noise

Pada bagian penutup ini, kita mempertemukan kedua algoritma dalam satu ruang pengujian yang adil (*apple-to-apple comparison*):
1. Menjalankan K-Means dan DBSCAN pada **10.000 sampel data yang identik**.
2. Menyajikan **tabel komparasi metrik menyeluruh** (Silhouette, DBI, ARI vs `kelas_harga`, dan rasio Noise).
3. Memvisualisasikan perbandingan proyeksi 2D PCA berdampingan.
4. Melakukan **inspeksi karakteristik properti yang diklasifikasikan sebagai Noise (`-1`)** oleh DBSCAN untuk memverifikasi apakah titik tersebut benar-benar properti unik/mewah ekstrem secara bisnis properti.

In [ ]:
# ==============================================================================
# BAGIAN 4.1: EVALUASI KOMPARATIF K-MEANS VS DBSCAN PADA SUBSAMPEL IDENTIK
# ==============================================================================
# Latih K-Means pada subsampel 10.000 baris identik dengan K optimal
kmeans_sub = KMeans(n_clusters=k_optimal, n_init=10, random_state=42)
df_sub['cluster_kmeans'] = kmeans_sub.fit_predict(X_scaled_sub)

# 1. Metrik K-Means
sil_km_sub = silhouette_score(X_scaled_sub, df_sub['cluster_kmeans'])
dbi_km_sub = davies_bouldin_score(X_scaled_sub, df_sub['cluster_kmeans'])
ari_km_sub = adjusted_rand_score(df_sub['kelas_harga'], df_sub['cluster_kmeans'])

# 2. Metrik DBSCAN (Non-Noise)
mask_db_valid = df_sub['cluster_dbscan'] != -1
n_cl_db = len(set(df_sub.loc[mask_db_valid, 'cluster_dbscan']))
if n_cl_db >= 2 and np.sum(mask_db_valid) > n_cl_db:
    sil_db_sub = silhouette_score(X_scaled_sub[mask_db_valid], df_sub.loc[mask_db_valid, 'cluster_dbscan'])
    dbi_db_sub = davies_bouldin_score(X_scaled_sub[mask_db_valid], df_sub.loc[mask_db_valid, 'cluster_dbscan'])
else:
    sil_db_sub = np.nan
    dbi_db_sub = np.nan
ari_db_sub = adjusted_rand_score(df_sub['kelas_harga'], df_sub['cluster_dbscan'])

komparasi_data = [
    {
        'Algoritma': f'K-Means (K={k_optimal})',
        'Jumlah_Klaster': k_optimal,
        'Noise_Persen': '0.00%',
        'Silhouette_Score': round(sil_km_sub, 4),
        'Davies_Bouldin_Index': round(dbi_km_sub, 4),
        'ARI_vs_kelas_harga': round(ari_km_sub, 4)
    },
    {
        'Algoritma': f'DBSCAN (eps={best_eps:.3f}, ms={best_ms})',
        'Jumlah_Klaster': n_cl_db,
        'Noise_Persen': f"{(~mask_db_valid).mean()*100:.2f}%",
        'Silhouette_Score': round(sil_db_sub, 4) if not np.isnan(sil_db_sub) else None,
        'Davies_Bouldin_Index': round(dbi_db_sub, 4) if not np.isnan(dbi_db_sub) else None,
        'ARI_vs_kelas_harga': round(ari_db_sub, 4)
    }
]

df_komparasi = pd.DataFrame(komparasi_data)
print("=" * 80)
print("TABEL KOMPARASI METRIK KLASTERISASI: K-MEANS VS DBSCAN (10.000 SAMPEL)")
print("=" * 80)
print(df_komparasi.to_string(index=False))
print("=" * 80)

In [ ]:
# ==============================================================================
# BAGIAN 4.2: VISUALISASI PROYEKSI PCA 2D BERDAMPINGAN (K-MEANS VS DBSCAN)
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel Kiri: K-Means
scatter_km = axes[0].scatter(
    X_pca_sub[:, 0], X_pca_sub[:, 1],
    c=df_sub['cluster_kmeans'],
    cmap='tab10',
    alpha=0.4,
    s=15
)
axes[0].set_title(f"K-Means (K={k_optimal}) — Partisi Sferis Seluruh Titik", fontsize=12, fontweight='bold')
axes[0].set_xlabel(f"PC1 ({var_ratio[0]*100:.1f}%)", fontsize=11)
axes[0].set_ylabel(f"PC2 ({var_ratio[1]*100:.1f}%)", fontsize=11)
plt.colorbar(scatter_km, ax=axes[0], label='Klaster ID')

# Panel Kanan: DBSCAN
axes[1].scatter(
    X_pca_sub[mask_db_valid, 0], X_pca_sub[mask_db_valid, 1],
    c=df_sub.loc[mask_db_valid, 'cluster_dbscan'],
    cmap='tab10',
    alpha=0.5,
    s=15,
    label='Area Padat (Klaster)'
)
axes[1].scatter(
    X_pca_sub[~mask_db_valid, 0], X_pca_sub[~mask_db_valid, 1],
    c='black',
    marker='x',
    s=25,
    alpha=0.6,
    label='Noise / Outlier (-1)'
)
axes[1].set_title(f"DBSCAN — Pemisahan Area Padat & Deteksi Noise (-1)", fontsize=12, fontweight='bold')
axes[1].set_xlabel(f"PC1 ({var_ratio[0]*100:.1f}%)", fontsize=11)
axes[1].set_ylabel(f"PC2 ({var_ratio[1]*100:.1f}%)", fontsize=11)
axes[1].legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# BAGIAN 4.3: PROFILING KARAKTERISTIK TITIK NOISE (-1) DBSCAN
# ==============================================================================
df_noise = df_sub[df_sub['cluster_dbscan'] == -1]
df_clustered = df_sub[df_sub['cluster_dbscan'] != -1]

profil_noise = pd.DataFrame([
    {
        'Kelompok': 'Titik Terklaster (Clustered)',
        'Jumlah_Unit': len(df_clustered),
        'Persen_Total': f"{len(df_clustered)/len(df_sub)*100:.1f}%",
        'Median_Harga_Rp': f"Rp {df_clustered['harga'].median():,.0f}",
        'Median_Luas_Tanah_m2': f"{df_clustered['luas_m2'].median():.1f}",
        'Median_Luas_Bgn_m2': f"{df_clustered['luas_bangunan'].median():.1f}",
        'Median_Kamar': f"{df_clustered['jumlah_kamar'].median():.0f}",
        'Persen_Komersil': f"{df_clustered['is_komersil'].mean()*100:.1f}%",
        'Proporsi_Kelas_Mahal': f"{(df_clustered['kelas_harga'] == 'Mahal').mean()*100:.1f}%"
    },
    {
        'Kelompok': 'Titik Noise (-1)',
        'Jumlah_Unit': len(df_noise),
        'Persen_Total': f"{len(df_noise)/len(df_sub)*100:.1f}%",
        'Median_Harga_Rp': f"Rp {df_noise['harga'].median():,.0f}",
        'Median_Luas_Tanah_m2': f"{df_noise['luas_m2'].median():.1f}",
        'Median_Luas_Bgn_m2': f"{df_noise['luas_bangunan'].median():.1f}",
        'Median_Kamar': f"{df_noise['jumlah_kamar'].median():.0f}",
        'Persen_Komersil': f"{df_noise['is_komersil'].mean()*100:.1f}%",
        'Proporsi_Kelas_Mahal': f"{(df_noise['kelas_harga'] == 'Mahal').mean()*100:.1f}%"
    }
])

print("=" * 80)
print("INSPEKSI KARAKTERISTIK TITIK NOISE DBSCAN VS TITIK TERKLASTER:")
print("=" * 80)
print(profil_noise.to_string(index=False))
print("=" * 80)

## Diskusi, Analisis Komparatif, dan Kesimpulan JS04

### 1. Sintesis Temuan K-Means vs DBSCAN
Berdasarkan hasil eksperimen pada dataset properti multi-kota Indonesia, terbukti bahwa kedua algoritma memiliki paradigma yang sangat berlainan:
1. **K-Means (Partisi Sferis Global):**
   - Mengasumsikan klaster berbentuk sferis/konveks dengan ukuran dan densitas seragam.
   - Kelebihan: Mampu memproses 56.170 baris secara cepat dengan kompleksitas linear $\mathcal{O}(k \cdot n \cdot d)$.
   - Kelemahan: Memaksa **setiap baris data masuk ke dalam salah satu klaster**, sehingga titik ekstrem/mewah (misalnya properti seharga puluhan miliar dengan luas tanah ribuan m²) ikut menarik posisi centroid dan mendistorsi profil klaster reguler.
2. **DBSCAN (Klasterisasi Spasial Berbasis Kepadatan):**
   - Tidak memerlukan penetapan jumlah klaster $K$ di awal dan mampu menemukan klaster dengan bentuk arbitrer.
   - Kelebihan: Secara elegan memisahkan titik-titik anomali (*outlier*) ke dalam label `-1` (Noise). Terbukti secara empiris pada Bagian 4.3 bahwa kelompok Noise memiliki median harga dan luas tanah yang jauh melampaui kelompok terklaster, mencerminkan properti mewah super-premium atau properti komersial unik yang memang terisolasi dari kepadatan pasar umum.
   - Kelemahan: Memiliki kompleksitas $\mathcal{O}(N^2)$ dan sangat sensitif terhadap parameter tunggal $\varepsilon$, di mana kerapatan pasar properti multi-kota (misalnya Jabodetabek yang sangat padat vs kota lapis kedua/ketiga yang renggang) sulit ditangkap secara optimal oleh satu nilai $\varepsilon$ seragam.

### 2. Kesiapan Menuju JS05 (Klasterisasi Hierarki & HDBSCAN)
Keterbatasan DBSCAN dengan nilai `eps` tunggal inilah yang menjadi motivasi utama transisi ke **Jobsheet 05**, di mana kita akan mempelajari **Klasterisasi Hierarki (Agglomerative)** untuk melihat pohon keterhubungan antar-segmen, serta **HDBSCAN** yang secara adaptif dapat mengekstraksi klaster dengan kepadatan bertingkat (*varying densities*).